# 🟡 MODELO 5: Bidirectional GRU (Alternativa eficiente)

## 📋 Objetivo del ejercicio
Usar **GRU** en lugar de LSTM para obtener modelo más **rápido** y **ligero**.

## ⚠️ Problema del LSTM
- LSTM tiene **4 gates** (input, forget, output, cell)
- Muchos parámetros → **lento** y necesita **más datos**

## ✅ Solución: GRU (Gated Recurrent Unit)
- Solo **2 gates** (reset, update)
- **~25% menos parámetros** que LSTM
- **Más rápido** de entrenar
- **Rendimiento similar** en muchas tareas

### 📚 Teoría de examen:
> **LSTM vs GRU:**
> 
> | Característica | LSTM | GRU |
> |---|---|---|
> | Gates | 4 | 2 |
> | Parámetros | Más | ~75% del LSTM |
> | Velocidad | Más lento | Más rápido |
> | Memoria | Mejor para secuencias muy largas | Suficiente para la mayoría |
> | Cuándo usar | Muchos datos, secuencias largas | Pocos datos, necesitas velocidad |

---

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

seed = 42
np.random.seed(seed)

In [ ]:
train = pd.read_csv("/kaggle/input/u-tad-spam-not-spam-2025-edition/train.csv", index_col="row_id")
test = pd.read_csv("/kaggle/input/u-tad-spam-not-spam-2025-edition/test.csv", index_col="row_id")

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf

tf.random.set_seed(seed)

X = train['message'].values
y = train['spam_label'].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)

MAX_WORDS = 5000
MAX_LEN = 100
EMBEDDING_DIM = 128

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN, padding='post')
X_val_pad = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=MAX_LEN, padding='post')
X_test_pad = pad_sequences(tokenizer.texts_to_sequences(test['message'].values), maxlen=MAX_LEN, padding='post')

print("✅ Datos preparados")

## 🏗️ Modelo Bidirectional GRU

### 📚 ¿Cómo funciona GRU?

**GRU simplifica LSTM:**
- **LSTM**: input gate + forget gate + output gate + cell state
- **GRU**: reset gate + update gate (combina input y forget)

**Reset gate**: Decide qué información del pasado olvidar
**Update gate**: Decide cuánta información nueva añadir

### 🔑 Pregunta de examen típica:
**P: ¿Cuándo elegir GRU sobre LSTM?**

**R:**
- ✅ **Dataset pequeño/mediano**: GRU necesita menos datos
- ✅ **Necesitas velocidad**: GRU entrena 20-30% más rápido
- ✅ **Secuencias cortas/medianas**: GRU suficiente
- ✅ **Recursos limitados**: GRU usa menos memoria

- ❌ **Dataset muy grande**: LSTM puede aprovechar mejor
- ❌ **Secuencias muy largas** (>500 tokens): LSTM mejor memoria
- ❌ **Máxima capacidad**: LSTM tiene más parámetros

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout, Bidirectional, LayerNormalization

model_gru = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    Dropout(0.3),
    
    # ✨ CAMBIO: GRU en lugar de LSTM
    Bidirectional(GRU(64, return_sequences=True)),
    LayerNormalization(),
    Dropout(0.4),
    
    Bidirectional(GRU(32)),
    LayerNormalization(),
    Dropout(0.4),
    
    Dense(32, activation='relu'),
    LayerNormalization(),
    Dropout(0.3),
    
    Dense(1, activation='sigmoid')
])

model_gru.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_gru.summary()

### 📊 Comparación de parámetros:

**Arquitectura idéntica, solo cambia LSTM → GRU**

Observa en el summary que GRU tiene **menos parámetros** que LSTM.

Ejemplo típico:
- LSTM(64): ~33K params
- GRU(64): ~25K params
- Reducción: ~25%

In [ ]:
import time

# Medir tiempo de entrenamiento
start_time = time.time()

history = model_gru.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=15,
    batch_size=32,
    verbose=1
)

training_time = time.time() - start_time
print(f"\n⏱️ Tiempo de entrenamiento: {training_time:.2f} segundos")
print(f"⏱️ Tiempo por época: {training_time/15:.2f} segundos")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('GRU Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('GRU Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
val_loss, val_acc = model_gru.evaluate(X_val_pad, y_val, verbose=0)
print(f"\n📊 Validation Accuracy: {val_acc:.4f}")
print(f"📊 Validation Loss: {val_loss:.4f}")
print(f"\n🔍 Compara este resultado con el modelo LSTM")
print(f"   Normalmente: accuracy similar, pero GRU más rápido")

In [ ]:
print(f"\n📊 Parámetros del modelo GRU: {model_gru.count_params():,}")
print(f"\n💡 Para comparar con LSTM: entrena modelo_4_bidirectional_lstm.ipynb")
print(f"   y compara número de parámetros y tiempo de entrenamiento")

In [ ]:
y_pred_proba = model_gru.predict(X_test_pad)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()

submission = pd.read_csv("/kaggle/input/u-tad-spam-not-spam-2025-edition/sample_submission.csv")
submission["spam_label"] = y_pred
submission.to_csv('submission_gru.csv', index=False)

print("✅ Submission creado: submission_gru.csv")

---

## 📚 Resumen para Examen

### ✅ Ventajas GRU:
- **25% menos parámetros** que LSTM
- **20-30% más rápido** de entrenar
- **Menos propenso a overfitting** con pocos datos
- **Rendimiento similar** a LSTM en la mayoría de casos
- Más simple de entender e implementar

### ❌ Desventajas GRU:
- **Ligeramente peor** en secuencias muy largas
- **Menor capacidad** de memoria a largo plazo
- En datasets gigantes, LSTM puede superarlo

### 🎯 Tabla de decisión LSTM vs GRU:

| Situación | Elección |
|-----------|----------|
| Dataset pequeño (<10K muestras) | **GRU** |
| Dataset grande (>100K muestras) | **LSTM** |
| Secuencias cortas (<100 tokens) | **GRU** |
| Secuencias largas (>500 tokens) | **LSTM** |
| Necesitas velocidad | **GRU** |
| Necesitas máxima accuracy | **LSTM** |
| Recursos limitados | **GRU** |
| Producción (inferencia rápida) | **GRU** |
| No estás seguro | **Prueba ambos y compara** |

### 💡 Consejo para examen:
> "En la duda, GRU es una opción segura: más rápido, menos parámetros,
> y rendimiento similar a LSTM en la mayoría de casos prácticos."